# Build Train / Val / Test Splits

Builds all 36 cell CSVs + 2 RFP window CSVs under `data/splits/` from raw
price series in `data/daily/` and `data/weekly/`, using boundaries defined in
`data/splits/manifest.yaml`.

See `data/splits/explanation.md` for the full splitting strategy and feature
reference.

Run this notebook end-to-end to (re)generate splits. All outputs are
deterministic.


In [1]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import random

ROOT = Path.cwd()
DATA = ROOT / "data"
SPLITS = DATA / "splits"
RAW = {"daily": DATA / "daily", "weekly": DATA / "weekly"}

with open(SPLITS / "manifest.yaml") as f:
    cfg = yaml.safe_load(f)

cfg


{'sample_start': datetime.date(2004, 1, 1),
 'splits': {'daily': {'train': [datetime.date(2004, 1, 1),
    datetime.date(2021, 12, 31)],
   'val': [datetime.date(2022, 1, 1), datetime.date(2023, 12, 31)],
   'test': [datetime.date(2024, 1, 1), datetime.date(2026, 4, 29)]},
  'weekly': {'train': [datetime.date(2004, 1, 2), datetime.date(2018, 12, 31)],
   'val': [datetime.date(2019, 1, 1), datetime.date(2019, 12, 31)],
   'test': [datetime.date(2020, 1, 1), datetime.date(2026, 4, 29)]}},
 'targets': ['SPY', 'OIL', 'GOLD'],
 'exog_pool': ['SPY', 'OIL', 'GOLD', 'VIX', 'DXY'],
 'embargo_periods': 1,
 'rfp': {'regimes': {'GFC': [datetime.date(2007, 7, 1),
    datetime.date(2009, 6, 30)],
   'OIL_CRASH': [datetime.date(2014, 6, 1), datetime.date(2016, 2, 29)],
   'COVID': [datetime.date(2020, 2, 1), datetime.date(2020, 12, 31)],
   'ENERGY_22': [datetime.date(2022, 1, 1), datetime.date(2023, 6, 30)],
   'CALM_17_19': [datetime.date(2017, 1, 1), datetime.date(2019, 12, 31)]},
  'daily': {'n_p

## 1. Load and align raw price series

The five raw weekly CSVs are anchored on different days of the week (SPY/VIX/DXY
on Saturdays, OIL/GOLD on Mondays), so a naive inner join yields zero rows. We
normalize each weekly date to the **Friday of its ISO week** before joining.
Daily series are joined directly on the trading date.

We load both `Close` and `Volume` columns. Volume is only used for SPY, OIL,
and GOLD (VIX has no volume; DXY volume is unreliable). Zero-volume days for
OIL and GOLD (data artifacts in futures contracts) are forward-filled before
taking the log.


In [2]:
SERIES = ["SPY", "OIL", "GOLD", "VIX", "DXY"]
ASSETS_WITH_VOLUME = ("SPY", "OIL", "GOLD")

def load_close_volume(path: Path, freq: str) -> pd.DataFrame:
    df = pd.read_csv(path, skiprows=[1, 2], header=0).rename(columns={"Price": "Date"})
    df["Date"] = pd.to_datetime(df["Date"])
    df = df[["Date", "Close", "Volume"]].set_index("Date").astype(float).sort_index()
    if freq == "weekly":
        df.index = df.index.to_period("W-FRI").to_timestamp(how="end").normalize()
        df = df[~df.index.duplicated(keep="last")]
    return df

def load_panel(freq: str):
    raw = {s: load_close_volume(RAW[freq] / f"{s}_{freq}.csv", freq) for s in SERIES}
    closes = pd.concat({s: raw[s]["Close"] for s in SERIES}, axis=1).dropna(how="any")
    volumes = pd.concat({s: raw[s]["Volume"] for s in ASSETS_WITH_VOLUME}, axis=1).reindex(closes.index)
    closes = closes.loc[closes.index >= pd.Timestamp(cfg["sample_start"])]
    volumes = volumes.loc[volumes.index >= pd.Timestamp(cfg["sample_start"])]
    closes.index.name = "date"
    volumes.index.name = "date"
    return closes, volumes

panels = {freq: load_panel(freq) for freq in ("daily", "weekly")}
for freq, (c, v) in panels.items():
    zeros = (v == 0).sum().to_dict()
    print(f"{freq:6s}  n={len(c):5d}  range={c.index.min().date()} -> {c.index.max().date()}  vol_zeros={zeros}")


daily   n= 5603  range=2004-01-05 -> 2026-04-29  vol_zeros={'SPY': 0, 'OIL': 6, 'GOLD': 168}
weekly  n= 1166  range=2004-01-02 -> 2026-05-01  vol_zeros={'SPY': 0, 'OIL': 0, 'GOLD': 6}


/tmp/ipykernel_2099/4063648930.py:15: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  closes = pd.concat({s: raw[s]["Close"] for s in SERIES}, axis=1).dropna(how="any")
/tmp/ipykernel_2099/4063648930.py:16: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  volumes = pd.concat({s: raw[s]["Volume"] for s in ASSETS_WITH_VOLUME}, axis=1).reindex(closes.index)


## 2. Build feature panels

For each (frequency, asset), compute a full set of causally-lagged features.

**Endogenous (target-only) features per asset S in {SPY, OIL, GOLD}:**

Return-based:
- `{S}_ret`               — log-return
- `{S}_ret_lag1`          — lagged log-return
- `{S}_ret_sq_lag1`       — lagged squared log-return (ARCH innovation)
- `{S}_neg_ret_sq_lag1`   — lagged negative-return squared (leverage indicator)
- `{S}_RV_{5,10,22}_lag1` — multi-horizon rolling realized vol (lagged)

Volume-based:
- `{S}_logvol_lag1`       — lagged log-volume
- `{S}_logvol_z22_lag1`   — lagged 22-period z-score of log-volume (detrended)
- `{S}_absret_logvol_lag1` — lagged |return| × log-volume (MDH interaction)

For DXY (price-only, no volume): `DXY_ret`, `DXY_ret_lag1`.
For VIX: `vix_log`, `vix_log_lag1`, `vix_log_diff_lag1` (level and change).


In [3]:
RV_WINDOWS = (5, 10, 22)
VOL_Z_WINDOW = 22

def build_features(closes: pd.DataFrame, volumes: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=closes.index)
    for s in ["SPY", "OIL", "GOLD", "DXY"]:
        ret = np.log(closes[s]).diff()
        out[f"{s}_ret"] = ret
        out[f"{s}_ret_lag1"] = ret.shift(1)
        if s == "DXY":
            continue
        # Return-based endogenous features
        ret_sq = ret ** 2
        out[f"{s}_ret_sq_lag1"] = ret_sq.shift(1)
        out[f"{s}_neg_ret_sq_lag1"] = ((ret < 0) * ret_sq).shift(1)
        for w in RV_WINDOWS:
            out[f"{s}_RV_{w}_lag1"] = ret.rolling(w).std().shift(1)
        # Volume-based endogenous features (zero-volume days forward-filled)
        vol_safe = volumes[s].replace(0, np.nan).ffill()
        logvol = np.log(vol_safe)
        out[f"{s}_logvol_lag1"] = logvol.shift(1)
        mu = logvol.rolling(VOL_Z_WINDOW).mean()
        sd = logvol.rolling(VOL_Z_WINDOW).std()
        out[f"{s}_logvol_z22_lag1"] = ((logvol - mu) / sd).shift(1)
        out[f"{s}_absret_logvol_lag1"] = (ret.abs() * logvol).shift(1)
    # Market-level features
    out["vix_log"] = np.log(closes["VIX"])
    out["vix_log_lag1"] = out["vix_log"].shift(1)
    out["vix_log_diff_lag1"] = out["vix_log"].diff().shift(1)
    return out.dropna()

features = {freq: build_features(c, v) for freq, (c, v) in panels.items()}
for freq, f in features.items():
    print(f"{freq:6s}  n={len(f):5d}  range={f.index.min().date()} -> {f.index.max().date()}  cols={len(f.columns)}")
features["daily"].head(3)


daily   n= 5556  range=2004-02-06 -> 2026-04-29  cols=35
weekly  n= 1143  range=2004-06-11 -> 2026-05-01  cols=35


/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,SPY_ret,SPY_ret_lag1,SPY_ret_sq_lag1,SPY_neg_ret_sq_lag1,SPY_RV_5_lag1,SPY_RV_10_lag1,SPY_RV_22_lag1,SPY_logvol_lag1,SPY_logvol_z22_lag1,SPY_absret_logvol_lag1,...,GOLD_RV_10_lag1,GOLD_RV_22_lag1,GOLD_logvol_lag1,GOLD_logvol_z22_lag1,GOLD_absret_logvol_lag1,DXY_ret,DXY_ret_lag1,vix_log,vix_log_lag1,vix_log_diff_lag1
date,,,,,,,,,,,,,,,,,,,,,
2004-02-06,0.011159,0.002920,8.525257e-06,0.0,0.004895,0.007328,0.006378,17.432539,0.161599,0.050900,...,0.014622,0.012424,4.875197,0.286042,0.035385,-0.010178,0.000691,2.772589,2.874129,-0.008994
2004-02-09,0.000262,0.011159,1.245201e-04,0.0,0.007198,0.008284,0.006786,17.432249,0.066091,0.194524,...,0.015498,0.012916,5.746203,0.506919,0.078844,-0.000232,-0.010178,2.796671,2.772589,-0.101541
2004-02-10,0.003226,0.000262,6.868281e-08,0.0,0.007058,0.007046,0.006761,17.028421,-1.537868,0.004463,...,0.015723,0.013091,4.615121,0.148410,0.036447,-0.001862,-0.000232,2.768832,2.796671,0.024083


## 3. Build per-cell tables

For each (frequency, exog_flag, target) cell:

- **`no_exog`**: target's own endogenous features only — return-based plus
  volume-based (the target's own volume features).
- **`with_exog`**: `no_exog` + cross-asset return lags (other prices in
  {SPY, OIL, GOLD}, target excluded) + DXY return lag + VIX level + VIX
  change.

Cross-asset *volume* features are intentionally not included — empirical
evidence for own-asset volume → own-asset volatility is strong, while
cross-asset volume effects are weaker and noisier.

Output column `ret` is the target's log-return (the y variable).


In [4]:
def build_cell(freq: str, target: str, use_exog: bool) -> pd.DataFrame:
    f = features[freq]
    cols = {"ret": f[f"{target}_ret"]}
    # --- Endogenous: return-based ---
    cols["ret_lag1"]         = f[f"{target}_ret_lag1"]
    cols["ret_sq_lag1"]      = f[f"{target}_ret_sq_lag1"]
    cols["neg_ret_sq_lag1"]  = f[f"{target}_neg_ret_sq_lag1"]
    for w in RV_WINDOWS:
        cols[f"RV_{w}_lag1"] = f[f"{target}_RV_{w}_lag1"]
    # --- Endogenous: volume-based (own-asset only) ---
    cols["logvol_lag1"]        = f[f"{target}_logvol_lag1"]
    cols["logvol_z22_lag1"]    = f[f"{target}_logvol_z22_lag1"]
    cols["absret_logvol_lag1"] = f[f"{target}_absret_logvol_lag1"]
    if use_exog:
        # Cross-asset return lags (target excluded)
        for s in ["SPY", "OIL", "GOLD"]:
            if s == target:
                continue
            cols[f"{s}_ret_lag1"] = f[f"{s}_ret_lag1"]
        # External market features
        cols["DXY_ret_lag1"]      = f["DXY_ret_lag1"]
        cols["vix_log_lag1"]      = f["vix_log_lag1"]
        cols["vix_log_diff_lag1"] = f["vix_log_diff_lag1"]
    df = pd.DataFrame(cols, index=f.index).dropna()
    df.index.name = "date"
    return df.reset_index()

print("daily/no_exog/SPY  cols:", list(build_cell("daily", "SPY", False).columns))
print("daily/with_exog/SPY cols:", list(build_cell("daily", "SPY", True).columns))
build_cell("daily", "SPY", True).head(3)


daily/no_exog/SPY  cols: ['date', 'ret', 'ret_lag1', 'ret_sq_lag1', 'neg_ret_sq_lag1', 'RV_5_lag1', 'RV_10_lag1', 'RV_22_lag1', 'logvol_lag1', 'logvol_z22_lag1', 'absret_logvol_lag1']
daily/with_exog/SPY cols: ['date', 'ret', 'ret_lag1', 'ret_sq_lag1', 'neg_ret_sq_lag1', 'RV_5_lag1', 'RV_10_lag1', 'RV_22_lag1', 'logvol_lag1', 'logvol_z22_lag1', 'absret_logvol_lag1', 'OIL_ret_lag1', 'GOLD_ret_lag1', 'DXY_ret_lag1', 'vix_log_lag1', 'vix_log_diff_lag1']


,date,ret,ret_lag1,ret_sq_lag1,neg_ret_sq_lag1,RV_5_lag1,RV_10_lag1,RV_22_lag1,logvol_lag1,logvol_z22_lag1,absret_logvol_lag1,OIL_ret_lag1,GOLD_ret_lag1,DXY_ret_lag1,vix_log_lag1,vix_log_diff_lag1
0,2004-02-06,0.011159,0.002920,8.525257e-06,0.0,0.004895,0.007328,0.006378,17.432539,0.161599,0.050900,-0.000604,-0.007258,0.000691,2.874129,-0.008994
1,2004-02-09,0.000262,0.011159,1.245201e-04,0.0,0.007198,0.008284,0.006786,17.432249,0.066091,0.194524,-0.018304,0.013721,-0.010178,2.772589,-0.101541
2,2004-02-10,0.003226,0.000262,6.868281e-08,0.0,0.007058,0.007046,0.006761,17.028421,-1.537868,0.004463,0.010718,0.007897,-0.000232,2.796671,0.024083


## 4. Slice into train / val / test and write CSVs

In [5]:
def slice_stage(df: pd.DataFrame, stage_dates) -> pd.DataFrame:
    start, end = pd.Timestamp(stage_dates[0]), pd.Timestamp(stage_dates[1])
    return df[(df["date"] >= start) & (df["date"] <= end)].reset_index(drop=True)

written = []
for freq in ("daily", "weekly"):
    for use_exog, sub in [(False, "no_exog"), (True, "with_exog")]:
        for target in cfg["targets"]:
            cell = build_cell(freq, target, use_exog)
            for stage, dates in cfg["splits"][freq].items():
                out = slice_stage(cell, dates)
                outdir = SPLITS / freq / sub / target
                outdir.mkdir(parents=True, exist_ok=True)
                out.to_csv(outdir / f"{stage}.csv", index=False, date_format="%Y-%m-%d")
                written.append((freq, sub, target, stage, len(out), out.shape[1]))

summary = pd.DataFrame(written, columns=["freq", "exog", "target", "stage", "rows", "cols"])
print(f"wrote {len(summary)} split files")
summary.pivot_table(index=["freq", "exog", "target"], columns="stage", values="rows")


wrote 36 split files


stage                     test   train    val
freq   exog      target                      
daily  no_exog   GOLD    583.0  4472.0  501.0
                 OIL     583.0  4472.0  501.0
                 SPY     583.0  4472.0  501.0
       with_exog GOLD    583.0  4472.0  501.0
                 OIL     583.0  4472.0  501.0
                 SPY     583.0  4472.0  501.0
weekly no_exog   GOLD    330.0   760.0   52.0
                 OIL     330.0   760.0   52.0
                 SPY     330.0   760.0   52.0
       with_exog GOLD    330.0   760.0   52.0
                 OIL     330.0   760.0   52.0
                 SPY     330.0   760.0   52.0

## 5. Generate RFP window definitions

For each (frequency, regime), draw up to `n_per_regime` non-overlapping windows
of length `window_len`, with each window's `forecast_end` strictly before the
frequency's `test_start`. Regimes whose entire span lies on/after `test_start`
are skipped for that frequency.


In [6]:
def generate_rfp(freq: str) -> pd.DataFrame:
    rfp_cfg = cfg["rfp"]
    freq_cfg = rfp_cfg[freq]
    n_per_regime = freq_cfg["n_per_regime"]
    window_len = freq_cfg["window_len"]
    embargo = cfg["embargo_periods"]
    test_start = pd.Timestamp(cfg["splits"][freq]["test"][0])
    rng = random.Random(rfp_cfg["random_seed"])
    prefix = "d" if freq == "daily" else "w"
    dates = pd.DatetimeIndex(features[freq].index)

    rows = []
    for regime_name, (rs, re_) in rfp_cfg["regimes"].items():
        rs, re_ = pd.Timestamp(rs), pd.Timestamp(re_)
        if re_ >= test_start:
            print(f"[{freq}] skip regime {regime_name}: span end {re_.date()} >= test_start {test_start.date()}")
            continue
        in_regime_pos = [dates.get_loc(d) for d in dates[(dates >= rs) & (dates <= re_)]]
        candidates = []
        for pos in in_regime_pos:
            if pos + window_len - 1 >= len(dates):
                continue
            if pos - embargo - 1 < 0:
                continue
            forecast_end = dates[pos + window_len - 1]
            if forecast_end >= test_start:
                continue
            candidates.append(pos)
        rng.shuffle(candidates)
        chosen = []
        for c in candidates:
            if all(abs(c - x) >= window_len for x in chosen):
                chosen.append(c)
            if len(chosen) == n_per_regime:
                break
        chosen.sort()
        if len(chosen) < n_per_regime:
            print(f"[{freq}] regime {regime_name}: produced {len(chosen)} non-overlapping windows (target {n_per_regime})")
        for k, start_pos in enumerate(chosen, start=1):
            rows.append({
                "window_id": f"{prefix}_{regime_name}_{k}",
                "regime": regime_name,
                "fit_end": dates[start_pos - embargo - 1].date().isoformat(),
                "forecast_start": dates[start_pos].date().isoformat(),
                "forecast_end": dates[start_pos + window_len - 1].date().isoformat(),
            })
    return pd.DataFrame(rows)

(SPLITS / "rfp").mkdir(parents=True, exist_ok=True)
rfp_dfs = {}
for freq in ("daily", "weekly"):
    df = generate_rfp(freq)
    df.to_csv(SPLITS / "rfp" / f"{freq}_windows.csv", index=False)
    rfp_dfs[freq] = df
    print(f"\n[{freq}] {len(df)} windows")
    print(df.to_string(index=False) if len(df) else "(none)")


[daily] regime COVID: produced 3 non-overlapping windows (target 5)
[daily] regime ENERGY_22: produced 4 non-overlapping windows (target 5)

[daily] 22 windows
     window_id     regime    fit_end forecast_start forecast_end
       d_GFC_1        GFC 2007-09-21     2007-09-25   2007-12-18
       d_GFC_2        GFC 2008-01-07     2008-01-09   2008-04-04
       d_GFC_3        GFC 2008-04-24     2008-04-28   2008-07-22
       d_GFC_4        GFC 2008-10-17     2008-10-21   2009-01-15
       d_GFC_5        GFC 2009-02-26     2009-03-02   2009-05-26
 d_OIL_CRASH_1  OIL_CRASH 2014-09-11     2014-09-15   2014-12-08
 d_OIL_CRASH_2  OIL_CRASH 2015-02-20     2015-02-24   2015-05-19
 d_OIL_CRASH_3  OIL_CRASH 2015-05-21     2015-05-26   2015-08-18
 d_OIL_CRASH_4  OIL_CRASH 2015-10-23     2015-10-27   2016-01-22
 d_OIL_CRASH_5  OIL_CRASH 2016-02-18     2016-02-22   2016-05-16
     d_COVID_1      COVID 2020-03-10     2020-03-12   2020-07-10
     d_COVID_2      COVID 2020-07-15     2020-07-17   2020-1

## 6. Verify outputs

In [7]:
all_csvs = sorted(SPLITS.rglob("*.csv"))
print(f"Total CSVs under data/splits/: {len(all_csvs)}")
for p in all_csvs:
    print(f"  {p.relative_to(ROOT)}")


Total CSVs under data/splits/: 38
  data/splits/daily/no_exog/GOLD/test.csv
  data/splits/daily/no_exog/GOLD/train.csv
  data/splits/daily/no_exog/GOLD/val.csv
  data/splits/daily/no_exog/OIL/test.csv
  data/splits/daily/no_exog/OIL/train.csv
  data/splits/daily/no_exog/OIL/val.csv
  data/splits/daily/no_exog/SPY/test.csv
  data/splits/daily/no_exog/SPY/train.csv
  data/splits/daily/no_exog/SPY/val.csv
  data/splits/daily/with_exog/GOLD/test.csv
  data/splits/daily/with_exog/GOLD/train.csv
  data/splits/daily/with_exog/GOLD/val.csv
  data/splits/daily/with_exog/OIL/test.csv
  data/splits/daily/with_exog/OIL/train.csv
  data/splits/daily/with_exog/OIL/val.csv
  data/splits/daily/with_exog/SPY/test.csv
  data/splits/daily/with_exog/SPY/train.csv
  data/splits/daily/with_exog/SPY/val.csv
  data/splits/rfp/daily_windows.csv
  data/splits/rfp/weekly_windows.csv
  data/splits/weekly/no_exog/GOLD/test.csv
  data/splits/weekly/no_exog/GOLD/train.csv
  data/splits/weekly/no_exog/GOLD/val.csv
  

In [8]:
import itertools
checks = []
for freq, sub, target, stage in itertools.product(
    ["daily", "weekly"], ["no_exog", "with_exog"], cfg["targets"], ["train", "val", "test"]
):
    p = SPLITS / freq / sub / target / f"{stage}.csv"
    df = pd.read_csv(p, parse_dates=["date"])
    expected = cfg["splits"][freq][stage]
    checks.append({
        "freq": freq, "exog": sub, "target": target, "stage": stage,
        "rows": len(df),
        "cols": df.shape[1],
        "first": df["date"].min().date() if len(df) else None,
        "last": df["date"].max().date() if len(df) else None,
        "expected": f"{expected[0]} to {expected[1]}",
    })
pd.DataFrame(checks)


,freq,exog,target,stage,rows,cols,first,last,expected
0,daily,no_exog,SPY,train,4472,11,2004-02-06,2021-12-31,2004-01-01 to 2021-12-31
1,daily,no_exog,SPY,val,501,11,2022-01-03,2023-12-29,2022-01-01 to 2023-12-31
2,daily,no_exog,SPY,test,583,11,2024-01-02,2026-04-29,2024-01-01 to 2026-04-29
3,daily,no_exog,OIL,train,4472,11,2004-02-06,2021-12-31,2004-01-01 to 2021-12-31
4,daily,no_exog,OIL,val,501,11,2022-01-03,2023-12-29,2022-01-01 to 2023-12-31
5,daily,no_exog,OIL,test,583,11,2024-01-02,2026-04-29,2024-01-01 to 2026-04-29
6,daily,no_exog,GOLD,train,4472,11,2004-02-06,2021-12-31,2004-01-01 to 2021-12-31
7,daily,no_exog,GOLD,val,501,11,2022-01-03,2023-12-29,2022-01-01 to 2023-12-31
8,daily,no_exog,GOLD,test,583,11,2024-01-02,2026-04-29,2024-01-01 to 2026-04-29
9,daily,with_exog,SPY,train,4472,16,2004-02-06,2021-12-31,2004-01-01 to 2021-12-31


In [9]:
# Schemas per cell type
for freq in ("daily", "weekly"):
    for sub in ("no_exog", "with_exog"):
        for target in cfg["targets"]:
            p = SPLITS / freq / sub / target / "train.csv"
            cols = tuple(pd.read_csv(p, nrows=0).columns)
            print(f"{freq:6s}/{sub:9s}/{target:4s} ({len(cols)} cols): {list(cols)}")


daily /no_exog  /SPY  (11 cols): ['date', 'ret', 'ret_lag1', 'ret_sq_lag1', 'neg_ret_sq_lag1', 'RV_5_lag1', 'RV_10_lag1', 'RV_22_lag1', 'logvol_lag1', 'logvol_z22_lag1', 'absret_logvol_lag1']
daily /no_exog  /OIL  (11 cols): ['date', 'ret', 'ret_lag1', 'ret_sq_lag1', 'neg_ret_sq_lag1', 'RV_5_lag1', 'RV_10_lag1', 'RV_22_lag1', 'logvol_lag1', 'logvol_z22_lag1', 'absret_logvol_lag1']
daily /no_exog  /GOLD (11 cols): ['date', 'ret', 'ret_lag1', 'ret_sq_lag1', 'neg_ret_sq_lag1', 'RV_5_lag1', 'RV_10_lag1', 'RV_22_lag1', 'logvol_lag1', 'logvol_z22_lag1', 'absret_logvol_lag1']
daily /with_exog/SPY  (16 cols): ['date', 'ret', 'ret_lag1', 'ret_sq_lag1', 'neg_ret_sq_lag1', 'RV_5_lag1', 'RV_10_lag1', 'RV_22_lag1', 'logvol_lag1', 'logvol_z22_lag1', 'absret_logvol_lag1', 'OIL_ret_lag1', 'GOLD_ret_lag1', 'DXY_ret_lag1', 'vix_log_lag1', 'vix_log_diff_lag1']
daily /with_exog/OIL  (16 cols): ['date', 'ret', 'ret_lag1', 'ret_sq_lag1', 'neg_ret_sq_lag1', 'RV_5_lag1', 'RV_10_lag1', 'RV_22_lag1', 'logvol_la

weekly/no_exog  /GOLD (11 cols): ['date', 'ret', 'ret_lag1', 'ret_sq_lag1', 'neg_ret_sq_lag1', 'RV_5_lag1', 'RV_10_lag1', 'RV_22_lag1', 'logvol_lag1', 'logvol_z22_lag1', 'absret_logvol_lag1']
weekly/with_exog/SPY  (16 cols): ['date', 'ret', 'ret_lag1', 'ret_sq_lag1', 'neg_ret_sq_lag1', 'RV_5_lag1', 'RV_10_lag1', 'RV_22_lag1', 'logvol_lag1', 'logvol_z22_lag1', 'absret_logvol_lag1', 'OIL_ret_lag1', 'GOLD_ret_lag1', 'DXY_ret_lag1', 'vix_log_lag1', 'vix_log_diff_lag1']
weekly/with_exog/OIL  (16 cols): ['date', 'ret', 'ret_lag1', 'ret_sq_lag1', 'neg_ret_sq_lag1', 'RV_5_lag1', 'RV_10_lag1', 'RV_22_lag1', 'logvol_lag1', 'logvol_z22_lag1', 'absret_logvol_lag1', 'SPY_ret_lag1', 'GOLD_ret_lag1', 'DXY_ret_lag1', 'vix_log_lag1', 'vix_log_diff_lag1']
weekly/with_exog/GOLD (16 cols): ['date', 'ret', 'ret_lag1', 'ret_sq_lag1', 'neg_ret_sq_lag1', 'RV_5_lag1', 'RV_10_lag1', 'RV_22_lag1', 'logvol_lag1', 'logvol_z22_lag1', 'absret_logvol_lag1', 'SPY_ret_lag1', 'OIL_ret_lag1', 'DXY_ret_lag1', 'vix_log_lag